# VideoMAE HF Model Evaluation (+ Optional Fine-Tuning)

**Problem fixed:** The old `scripts/run_videomae_eval.py` used the gait autoencoder
with MOG2 background subtraction on UCF-Crime videos — completely ignoring the
actual VideoMAE HuggingFace model at `models/videoMae/best_model/`.

This notebook:
1. Loads the real HF VideoMAE model (14-class ViT-B)
2. Evaluates it on the UCF-Crime test set
3. **If F1 < 0.70** — runs optional fine-tuning on UCF-Crime train split

Normal class id=7 (`Normal_Videos_Event`) per `config.json` id2label.

**Colab**: Runtime → T4 GPU (8–16 GB VRAM needed for fine-tuning)  
**Local**: Skip Drive mount cell.

In [1]:
!pip install transformers torch torchvision opencv-python-headless --quiet

In [ ]:
import sys
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    print('Drive mounted.')
else:
    print('Local mode.')

In [ ]:
import sys
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    # ── EDIT: path to your project on Google Drive ──
    PROJECT_ROOT = Path('/content/drive/MyDrive/majorProject')
else:
    PROJECT_ROOT = Path('..').resolve()

DATASET_DIR  = PROJECT_ROOT / 'datasets' / 'anomalydetectiondatasetucf'
MODEL_DIR    = PROJECT_ROOT / 'models' / 'videoMae' / 'best_model'
RESULTS_DIR  = PROJECT_ROOT / 'results'
NORMAL_CLASS = 7  # id2label[7] = 'Normal_Videos_Event'

assert DATASET_DIR.exists(), f'UCF-Crime not found: {DATASET_DIR}'
assert MODEL_DIR.exists(),   f'HF model not found: {MODEL_DIR}'
RESULTS_DIR.mkdir(exist_ok=True)
print('DATASET_DIR:', DATASET_DIR)
print('MODEL_DIR  :', MODEL_DIR)

In [ ]:
import torch
from transformers import VideoMAEForVideoClassification, VideoMAEImageProcessor

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')
if DEVICE == 'cuda':
    print(f'GPU  : {torch.cuda.get_device_name(0)}')
    print(f'VRAM : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')

processor = VideoMAEImageProcessor.from_pretrained(str(MODEL_DIR))
model     = VideoMAEForVideoClassification.from_pretrained(str(MODEL_DIR))
model.to(DEVICE).eval()
print(f'\nModel loaded — {model.config.num_labels} classes')
print('id2label:', model.config.id2label)

In [ ]:
import cv2, json
import numpy as np

def load_video_frames(video_path, num_frames=16):
    cap   = cv2.VideoCapture(str(video_path))
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if total < num_frames:
        cap.release(); return None
    indices = np.linspace(0, total - 1, num_frames, dtype=int)
    frames  = []
    for idx in indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(idx))
        ret, frame = cap.read()
        if not ret: cap.release(); return None
        frames.append(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    cap.release()
    return frames

def find_test_videos(dataset_dir):
    dataset_dir = Path(dataset_dir)
    anomaly = []
    tf = dataset_dir / 'Anomaly_Test.txt'
    if tf.exists():
        for line in open(tf):
            rel = line.strip()
            if rel:
                full = dataset_dir / rel
                if full.exists(): anomaly.append((full, 1))
    nd = dataset_dir / 'Normal_Videos_for_Event_Recognition' / 'Normal_Videos_for_Event_Recognition'
    normal = [(p, 0) for p in sorted(nd.glob('*.mp4'))] if nd.exists() else []
    return anomaly, normal

anomaly_vids, normal_vids = find_test_videos(DATASET_DIR)
print(f'Anomaly: {len(anomaly_vids)}  Normal: {len(normal_vids)}')

In [ ]:
rows, skipped = [], 0
all_vids = anomaly_vids + normal_vids

for i, (vp, label) in enumerate(all_vids):
    if (i+1) % 20 == 0: print(f'  [{i+1}/{len(all_vids)}]')
    frames = load_video_frames(vp)
    if frames is None: skipped += 1; continue
    try:
        inp = processor(list(frames), return_tensors='pt')
        inp = {k: v.to(DEVICE) for k, v in inp.items()}
        with torch.no_grad():
            probs = torch.softmax(model(**inp).logits, dim=-1)[0].cpu().numpy()
        rows.append({
            'video': vp.name, 'true_label': label,
            'anomaly_score': float(1.0 - probs[NORMAL_CLASS]),
            'normal_prob':   float(probs[NORMAL_CLASS]),
            'pred_class':    int(probs.argmax()),
        })
    except Exception as e:
        print(f'Error {vp.name}: {e}'); skipped += 1

scores = np.array([r['anomaly_score'] for r in rows])
labels = np.array([r['true_label']    for r in rows])

best_f1, best_t = 0.0, 0.5
for t in np.linspace(0.01, 0.99, 400):
    pp = (scores >= t).astype(int)
    tp = int(((pp==1)&(labels==1)).sum())
    fp = int(((pp==1)&(labels==0)).sum())
    fn = int(((pp==0)&(labels==1)).sum())
    p  = tp/(tp+fp+1e-9); r = tp/(tp+fn+1e-9)
    f1 = 2*p*r/(p+r+1e-9)
    if f1 > best_f1: best_f1, best_t = f1, float(t)

preds = (scores >= best_t).astype(int)
tp = int(((preds==1)&(labels==1)).sum())
fp = int(((preds==1)&(labels==0)).sum())
fn = int(((preds==0)&(labels==1)).sum())
tn = int(((preds==0)&(labels==0)).sum())
prec = tp/(tp+fp+1e-9); rec = tp/(tp+fn+1e-9)
f1   = 2*prec*rec/(prec+rec+1e-9); acc = (tp+tn)/len(rows)

metrics = {
    'best_threshold': best_t, 'precision': float(prec), 'recall': float(rec),
    'f1': float(f1), 'accuracy': float(acc),
    'tp': tp, 'fp': fp, 'fn': fn, 'tn': tn,
    'n_anomaly': int((labels==1).sum()), 'n_normal': int((labels==0).sum()),
    'n_total': len(rows), 'n_skipped': skipped,
    'normal_mean_score':  float(scores[labels==0].mean()) if (labels==0).any() else 0.0,
    'anomaly_mean_score': float(scores[labels==1].mean()) if (labels==1).any() else 0.0,
}
print(f'\nτ*={best_t:.4f}  Precision={prec:.4f}  Recall={rec:.4f}  F1={f1:.4f}  Acc={acc:.4f}')
print(f'TP={tp}  FP={fp}  FN={fn}  TN={tn}  Skipped={skipped}')
print(f'Normal mean score : {metrics["normal_mean_score"]:.4f}')
print(f'Anomaly mean score: {metrics["anomaly_mean_score"]:.4f}')

with open(RESULTS_DIR / 'videomae_hf_metrics.json', 'w') as f:
    json.dump(metrics, f, indent=2)
print(f'\nSaved → {RESULTS_DIR}/videomae_hf_metrics.json')
if f1 < 0.70:
    print('\n[WARNING] F1 < 0.70 — run the fine-tuning cells below.')

In [ ]:
import matplotlib.pyplot as plt

nm_sc = scores[labels==0]; abn_sc = scores[labels==1]
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.hist(nm_sc,  bins=30, alpha=0.6, color='blue', label=f'Normal (n={len(nm_sc)})')
ax1.hist(abn_sc, bins=30, alpha=0.6, color='red',  label=f'Anomaly (n={len(abn_sc)})')
ax1.axvline(best_t, color='black', linestyle='--', label=f'τ*={best_t:.3f}')
ax1.set_xlabel('Anomaly Score'); ax1.set_title('Score Distribution'); ax1.legend()

precs, recs = [], []
for t in np.linspace(0.01,0.99,200):
    pp = (scores>=t).astype(int)
    tp_ = int(((pp==1)&(labels==1)).sum())
    fp_ = int(((pp==1)&(labels==0)).sum())
    fn_ = int(((pp==0)&(labels==1)).sum())
    precs.append(tp_/(tp_+fp_+1e-9)); recs.append(tp_/(tp_+fn_+1e-9))
ax2.plot(recs, precs, color='purple')
ax2.scatter([rec], [prec], color='black', zorder=5, label=f'Best F1={f1:.3f}')
ax2.set_xlabel('Recall'); ax2.set_ylabel('Precision')
ax2.set_title('PR Curve — VideoMAE HF'); ax2.legend()
plt.tight_layout()
out = RESULTS_DIR / 'videomae_hf_pr_curve.png'
plt.savefig(str(out), dpi=120); plt.show(); print(f'Saved → {out}')

---
## Optional Fine-Tuning (only if F1 < 0.70)

Fine-tunes the **last 2 transformer blocks + classifier head** only.  
First 10 blocks frozen to preserve pre-trained representations.  
Requires ~8–12 GB VRAM (T4 or A100).

In [ ]:
from torch.utils.data import Dataset, DataLoader

class UCFCrimeDataset(Dataset):
    def __init__(self, entries, processor, num_frames=16):
        self.entries   = entries
        self.processor = processor
        self.nf        = num_frames
        self.label2id  = {v: int(k) for k, v in model.config.id2label.items()}

    def __len__(self): return len(self.entries)

    def __getitem__(self, idx):
        path, cls = self.entries[idx]
        frames    = load_video_frames(path, self.nf)
        if frames is None:
            frames = [np.zeros((224,224,3), dtype=np.uint8)] * self.nf
        inp   = self.processor(list(frames), return_tensors='pt')
        label = self.label2id.get(cls, self.label2id['Normal_Videos_Event'])
        return {k: v.squeeze(0) for k, v in inp.items()}, torch.tensor(label)

def build_train_entries(dataset_dir):
    entries = []
    for fname, cls_key in [('Anomaly_Train.txt', None), ('Normal_Train.txt', 'Normal_Videos_Event')]:
        tf = dataset_dir / fname
        if not tf.exists(): continue
        for line in open(tf):
            rel = line.strip()
            if not rel: continue
            full = dataset_dir / rel
            if full.exists():
                cls = cls_key or Path(rel).parts[-2]
                entries.append((full, cls))
    return entries

train_entries = build_train_entries(DATASET_DIR)
print(f'Fine-tune entries: {len(train_entries)}')

In [ ]:
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR

FT_EPOCHS = 5
FT_BATCH  = 4 if DEVICE == 'cuda' else 1

ft_model = VideoMAEForVideoClassification.from_pretrained(str(MODEL_DIR)).to(DEVICE)

# Freeze all, then unfreeze last 2 encoder blocks + classifier
for p in ft_model.parameters(): p.requires_grad = False
for name, p in ft_model.named_parameters():
    if any(f'encoder.layer.{i}.' in name for i in [10, 11]) or 'classifier' in name:
        p.requires_grad = True

trainable = sum(p.numel() for p in ft_model.parameters() if p.requires_grad)
print(f'Trainable: {trainable:,} params')

train_dl = DataLoader(UCFCrimeDataset(train_entries, processor), batch_size=FT_BATCH,
                      shuffle=True, num_workers=2)
opt   = AdamW(filter(lambda p: p.requires_grad, ft_model.parameters()), lr=2e-5)
sched = CosineAnnealingLR(opt, T_max=FT_EPOCHS, eta_min=1e-7)
loss_fn = torch.nn.CrossEntropyLoss()

for epoch in range(1, FT_EPOCHS+1):
    ft_model.train()
    losses = []
    for batch_inp, batch_labels in train_dl:
        batch_inp    = {k: v.to(DEVICE) for k, v in batch_inp.items()}
        batch_labels = batch_labels.to(DEVICE)
        loss = loss_fn(ft_model(**batch_inp).logits, batch_labels)
        opt.zero_grad(); loss.backward()
        torch.nn.utils.clip_grad_norm_(ft_model.parameters(), 1.0)
        opt.step(); losses.append(loss.item())
    sched.step()
    print(f'  Epoch {epoch}/{FT_EPOCHS}  loss={np.mean(losses):.4f}')

ft_out = PROJECT_ROOT / 'models' / 'videoMae' / 'best_model_finetuned'
ft_model.save_pretrained(str(ft_out))
processor.save_pretrained(str(ft_out))
print(f'\nFine-tuned model saved → {ft_out}')
print('Re-run eval cells with MODEL_DIR = ft_out to measure improvement.')